In [ ]:
#!/usr/bin/env python3
import os
import time
import json
import asyncio
import aiohttp
import requests
import pandas as pd
from tqdm import tqdm
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import nest_asyncio

nest_asyncio.apply()
# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY          = ""
OPENROUTER_URL   = "https://openrouter.ai/api/v1/chat/completions"
CSV_PATH         = "/content/bangla-med-qa.csv"
MODEL            = "qwen/qwen3.7-plus"  # change
OUTPUT_DIR       = "/content/drive/MyDrive/bangla_med_benchmark"
NUM_SAMPLES      = None                   # None = full dataset
CHECKPOINT_EVERY = 15
CONCURRENCY      = 5                   # parallel requests — tune per rate limit
BATCH_SIZE   = 15   # rows per batch
BATCH_DELAY  = 10   # seconds to wait between batches
SITE_URL  = "https://colab.research.google.com"
SITE_NAME = "Bangla Med Benchmark"

MAX_RETRIES     = 5
BASE_BACKOFF    = 2.0
REQUEST_TIMEOUT = 30

VALID_LETTERS = ["A", "B", "C", "D"]

SYSTEM_PROMPT = """You are answering a multiple-choice question. Output only the correct option letter(s). If there is one correct answer, output exactly one of A, B, C, or D. If there are multiple correct answers, output the letters in alphabetical order separated by commas (e.g., A,B or B,D or A,B,C). Output nothing else. Never give blank output."""

USER_TEMPLATE = """Question: {question}
A) {opt_a}
B) {opt_b}
C) {opt_c}
D) {opt_d}
Answer:"""


# ── Label helpers ─────────────────────────────────────────────────────────────
def normalize_label(raw):
    """Extract and sort all A/B/C/D letters into a canonical string e.g. 'AB', 'ACD'."""
    if raw is None:
        return None
    found = sorted(set(ch for ch in str(raw).upper() if ch in VALID_LETTERS))
    return "".join(found) if found else None


def parse_response(raw_text):
    return normalize_label(raw_text)


# ── API ───────────────────────────────────────────────────────────────────────
def build_headers():
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": SITE_URL,
        "X-Title": SITE_NAME,
    }


def build_user_message(row):
    return USER_TEMPLATE.format(
        question=row["question"],
        opt_a=row["options/A"],
        opt_b=row["options/B"],
        opt_c=row["options/C"],
        opt_d=row["options/D"],
    )


async def call_model_async(session, model, user_message, headers):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        "max_tokens": 10,
        "temperature": 0,
        "reasoning": {
          "effort": "none"
        },
    }
    start = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with session.post(
                OPENROUTER_URL, headers=headers, json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
            ) as resp:
                if resp.status == 429:
                    wait = float(resp.headers.get("Retry-After", BASE_BACKOFF * attempt))
                    await asyncio.sleep(wait)
                    continue
                if resp.status >= 500:
                    await asyncio.sleep(BASE_BACKOFF * attempt)
                    continue
                if resp.status != 200:
                    text = await resp.text()
                    return None, f"http_error:{resp.status}:{text[:200]}", time.monotonic() - start
                data    = await resp.json()
                content = data["choices"][0]["message"]["content"]
                return content, None, time.monotonic() - start
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start
            await asyncio.sleep(BASE_BACKOFF * attempt)

    return None, "max_retries_exceeded", time.monotonic() - start


# ── Checkpoint ────────────────────────────────────────────────────────────────
def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")

def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)


# ── Main loop ─────────────────────────────────────────────────────────────────
async def run_model_async(model, df, headers, out_dir):
    existing  = load_checkpoint(out_dir, model)
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows_done

    sem     = asyncio.Semaphore(CONCURRENCY)
    indices = list(range(start_idx, len(df)))
    batches = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]
    results = []

    async def process_row(session, i):
        async with sem:
            row      = df.iloc[i]
            label    = normalize_label(row["answer"])
            user_msg = build_user_message(row)

            raw, err, latency = await call_model_async(session, model, user_msg, headers)
            pred = parse_response(raw)

            # if pred is None:
            #     print(f"[{i}] pred=None (raw='{raw}', err='{err}') — retrying…")
            #     raw, err, latency = await call_model_async(session, model, user_msg, headers)
            #     pred = parse_response(raw)

            is_multi = len(label) > 1 if label else False
            correct  = (pred == label) if pred is not None else False
            print(f"[{i}] raw='{raw}' → pred={pred} | label={label} | multi={is_multi} | correct={correct}")

            return {
                "serial_no":    row.get("serial_no", i),
                "exam_name":    row.get("exam_name", ""),
                "question":     row["question"],
                "label":        label,
                "prediction":   pred,
                "is_multi":     is_multi,
                "correct":      correct,
                "latency":      latency,
                "raw_response": raw,
                "error":        err,
                "_idx":         i,
            }

    async with aiohttp.ClientSession() as session:
        for b_num, batch in enumerate(batches):
            print(f"\nBatch {b_num + 1}/{len(batches)} — rows {batch[0]}..{batch[-1]}")
            tasks        = [process_row(session, i) for i in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)

            all_rows = rows_done + sorted(results, key=lambda r: r["_idx"])
            save_checkpoint(out_dir, model, all_rows)

            if b_num < len(batches) - 1:
                print(f"Waiting {BATCH_DELAY}s before next batch…")
                await asyncio.sleep(BATCH_DELAY)

    results = sorted(results, key=lambda r: r["_idx"])
    for r in results:
        del r["_idx"]

    all_rows = rows_done + results
    save_checkpoint(out_dir, model, all_rows)
    return all_rows


    def run_model_on_dataset(model, df, headers, out_dir):
      loop = asyncio.get_event_loop()
      return loop.run_until_complete(run_model_async(model, df, headers, out_dir))


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(rows):
    """Compute metrics overall, and separately for single vs multi-answer rows."""

    def _metrics(subset):
        valid = [r for r in subset if r["prediction"] is not None and str(r["prediction"]).strip() not in ("", "nan")]
        if not valid:
            return {"accuracy": None, "precision": None, "recall": None, "f1": None, "valid": 0, "total": len(subset)}
        yt = [str(r["label"])      for r in valid]
        yp = [str(r["prediction"]) for r in valid]
        classes = sorted(set(yt) | set(yp))
        return {
            "accuracy":  accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, labels=classes, average="macro", zero_division=0),
            "recall":    recall_score(yt, yp,    labels=classes, average="macro", zero_division=0),
            "f1":        f1_score(yt, yp,        labels=classes, average="macro", zero_division=0),
            "valid":     len(valid),
            "total":     len(subset),
        }

    latencies   = [r["latency"] for r in rows if r["latency"] is not None]
    single_rows = [r for r in rows if not r["is_multi"]]
    multi_rows  = [r for r in rows if r["is_multi"]]

    return {
        "overall":       _metrics(rows),
        "single_answer": _metrics(single_rows),
        "multi_answer":  _metrics(multi_rows),
        "avg_latency":   sum(latencies) / len(latencies) if latencies else None,
    }


# ── Entry point ───────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    required = {"question", "options/A", "options/B", "options/C", "options/D", "answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required}")

    df["answer"] = df["answer"].apply(normalize_label)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"Dataset: {len(df)} rows")
    print(f"Single-answer: {(df['answer'].str.len() == 1).sum()} | Multi-answer: {(df['answer'].str.len() > 1).sum()}")

    headers = build_headers()
    rows    = run_model_on_dataset(MODEL, df, headers, OUTPUT_DIR)
    metrics = compute_metrics(rows)

    # ── Save predictions ──
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")

    # ── Save results ──
    results_rows = []
    for split, m in metrics.items():
        if split == "avg_latency":
            continue
        results_rows.append({
            "model":       MODEL,
            "split":       split,
            "accuracy":    m.get("accuracy"),
            "precision":   m.get("precision"),
            "recall":      m.get("recall"),
            "f1":          m.get("f1"),
            "valid":       m.get("valid"),
            "total":       m.get("total"),
            "avg_latency": metrics["avg_latency"],
        })

    results_df = pd.DataFrame(results_rows)
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

    print("\n── Results ──")
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: 994 rows
Single-answer: 985 | Multi-answer: 9

Batch 1/40 — rows 395..409
[399] raw='A' → pred=A | label=A | multi=False | correct=True
[395] raw='D' → pred=D | label=D | multi=False | correct=True
[400] raw='B' → pred=B | label=B | multi=False | correct=True
[398] raw='D' → pred=D | label=D | multi=False | correct=True
[397] raw='C' → pred=C | label=C | multi=False | correct=True
[401] raw='A' → pred=A | label=A | multi=False | correct=True
[403] raw='B' → pred=B | label=B | multi=False | correct=True
[405] raw='D' → pred=D | label=D | multi=False | correct=True
[406] raw='D' → pred=D | label=A | multi=False | correct=False
[396] raw='B' → pred=B | label=B | multi=False | correct=True
[408] raw='B' → pred=B | label=B | multi=False | correct=True
[404] raw='C' → pred=C | label=C | multi=False | correct=True
[407] raw='B' → pred=B | label=B | multi=Fa